In [ ]:
# Install library yang dibutuhkan
!pip install -q fastapi uvicorn ultralytics pyngrok python-multipart nest-asyncio

print("✅ Dependencies terinstall!")

In [ ]:
from google.colab import drive
import os
import sys

# 1. Mount Drive dengan force_remount agar pop-up selalu muncul jika token expired
print("🔐 Meminta akses Google Drive...")
drive.mount('/content/drive', force_remount=True)

# 2. Validasi path model SEBELUM server dinyalakan
MODEL_PATH = '/content/drive/MyDrive/Model/best.pt'
if not os.path.exists(MODEL_PATH):
    print(f"\n❌ FATAL: Model '{MODEL_PATH}' TIDAK DITEMUKAN!")
    print("Cek isi folder YoloProject di Drive Anda:")
    !ls "/content/drive/MyDrive/Model/"
    sys.exit("Hentikan eksekusi! Perbaiki path atau upload best.pt terlebih dahulu.")
else:
    print(f"✅ Model berhasil divalidasi di: {MODEL_PATH}")
    print("Silakan lanjutkan ke CELL 2 untuk menyalakan server.")

In [ ]:

# !ngrok config add-authtoken (ngrok-api-key)
# buka comment setelah paster api key

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading
import time
import os
import uuid
import io
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from ultralytics import YOLO
from PIL import Image

nest_asyncio.apply()

# ==========================================
# SETUP FASTAPI & RESIZE HELPER (ORIGINAL)
# ==========================================
app = FastAPI(title="CompostMind AI Core - Colab Edition")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def resize_to_800x800(image_bytes: bytes) -> bytes:
    """Versi original: Resize murni dengan PIL + Padding Hitam"""
    try:
        img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        target_size = 800
        scale = min(target_size / img.width, target_size / img.height)
        new_width = int(img.width * scale)
        new_height = int(img.height * scale)

        resized_img = img.resize((new_width, new_height), Image.LANCZOS)
        final_img = Image.new("RGB", (target_size, target_size), color=(0, 0, 0))
        paste_x = (target_size - new_width) // 2
        paste_y = (target_size - new_height) // 2
        final_img.paste(resized_img, (paste_x, paste_y))

        buffer = io.BytesIO()
        final_img.save(buffer, format="JPEG", quality=95)
        return buffer.getvalue()
    except Exception as e:
        print(f"️ Gagal resize: {e}. Menggunakan gambar asli.")
        return image_bytes

# ==========================================
# LOAD MODEL
# ==========================================
MODEL_PATH = '/content/drive/MyDrive/Model/best.pt'
print(f"⏳ Loading model dari Drive ke GPU...")
model = YOLO(MODEL_PATH)
model.to('cuda')
print("✅ Model loaded di GPU!")

os.makedirs("temp_uploads", exist_ok=True)

@app.post("/detect")
async def detect_compost(file: UploadFile = File(...)):
    if not file.content_type or not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="File harus berupa gambar.")

    original_bytes = await file.read()
    processed_bytes = resize_to_800x800(original_bytes)

    temp_filename = f"{uuid.uuid4()}.jpg"
    temp_filepath = os.path.join("temp_uploads", temp_filename)

    try:
        with open(temp_filepath, "wb") as buffer:
            buffer.write(processed_bytes)

        results = model(temp_filepath, conf=0.2, iou=0.2, verbose=False)
        detections = []

        for result in results:
            if result.boxes is not None:
                for i in range(len(result.boxes)):
                    class_id = int(result.boxes.cls[i].item())
                    class_name = model.names.get(class_id, "unknown")
                    confidence = float(result.boxes.conf[i].item())

                    # Filter manual (redundan tapi aman untuk versi original)
                    if confidence > 0.2:
                        detections.append({
                            "name": class_name,
                            "confidence": round(confidence, 2)
                        })

        return {"status": "success", "detections": detections}
    finally:
        if os.path.exists(temp_filepath):
            os.remove(temp_filepath)

# ==========================================
# NGROK & SERVER RUNNER
# ==========================================
try:
    ngrok.kill()
except:
    pass

os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(2)

public_url = ngrok.connect(8000)
print(f"\n🌍 PUBLIC URL: {public_url}")
print(f"📋 Copy URL ini ke Vercel env var PYTHON_AI_URL")
print(f"🧪 Test endpoint: {public_url}/detect")
print("\n⚠️ JANGAN TUTUP CELL INI selama presentasi!\n")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning", workers=1)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

while True:
    time.sleep(60)